## Опис

У цьому Jupyter Notebook описано процес підготовки даних для навчання моделі YOLO. Він включає такі етапи:

1. Парсинг наявної розмітки у форматі XML.
2. Створення власного розділення даних на тренувальну та валідаційну вибірки.
3. Збереження нової власної розмітки у форматі YOLO.
4. Створення кадрів з вихідних відео з врахуванням власного розбиття на тренувальну та валідаціну вибірки.
5. Створення конфігураційного файлу датасету YOLO (data.yaml).

## Імпорт бібліотек

In [6]:
import kagglehub
import os
import xml.etree.ElementTree as ET
import pandas as pd
import numpy as np
import cv2

In [7]:
import sys
sys.path.append('..')
from config import RANDOM_SEED 

rng = np.random.default_rng(seed=RANDOM_SEED) 

## Завантаження даних

In [8]:
path = kagglehub.dataset_download("swjtuziwei/fbd-sv-2024", output_dir='../data/raw')

100%|██████████| 1.68G/1.68G [00:50<00:00, 35.3MB/s]

Extracting files...


## Парсинг та аналіз розмітки

Оскільки початкова розмітка зберігається у форматі XML, а датасет уже має початковий поділ на тренувальну та валідаційну вибірки, спочатку необхідно розпарсити XML-файли та зібрати інформацію про всі анотовані кадри. Це дозволить потім створити власний поділ даних на тренувальну та валідаційну вибірик на рівні відео, не залежачи від початкового розподілу.

In [9]:
def parse_xml_labels(labels_path):

    data = []

    # папки train та val вихідних даних 
    for dataset in ['train', 'val']:

        folder_path = f'{labels_path}/{dataset}'

        for xml_file in os.listdir(folder_path):

            xml_path = f'{folder_path}/{xml_file}'

            tree = ET.parse(xml_path)
            root = tree.getroot()

            filename = root.find("filename").text
            _, video_number, frame_name = filename.split("_")

            width = int(root.find("size/width").text)
            height = int(root.find("size/height").text)
            depth = int(root.find("size/depth").text)


            objects = []

            for obj in root.findall("object"):

                bbox = obj.find("bndbox")

                objects.append({
                    "name": obj.find("name").text,
                    "difficult": int(obj.find("difficult").text),
                    "xmin": int(bbox.find("xmin").text),
                    "ymin": int(bbox.find("ymin").text),
                    "xmax": int(bbox.find("xmax").text),
                    "ymax": int(bbox.find("ymax").text)
                })

            data.append({
                "video_name": f'bird_{video_number}',
                "frame_name": f'bird_{video_number}_{frame_name}',
                "width": width,
                "height": height,
                "depth": depth,
                "objects": objects
            })


    return pd.DataFrame(data)

In [10]:
labels_df = parse_xml_labels('../data/raw/FBD-SV-2024/labels')

In [11]:
labels_df.head()

,video_name,frame_name,width,height,depth,objects
0,bird_100,bird_100_000000.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 1071..."
1,bird_100,bird_100_000001.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 1067..."
2,bird_100,bird_100_000002.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 1057..."
3,bird_100,bird_100_000003.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 1039..."
4,bird_100,bird_100_000004.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 2, 'xmin': 1024..."


In [12]:
video_count = len( labels_df["video_name"].unique() )
frame_count = len( labels_df["frame_name"].unique() )

print(f'Кількість відео: { video_count }')
print(f'Кількість фреймів: { frame_count }')
print(f'Середня кількість фреймів на одне відео: { frame_count / video_count }')

Кількість відео: 483
Кількість фреймів: 28694
Середня кількість фреймів на одне відео: 59.40786749482402


In [13]:
# розділення кількості об'єктів на одному фреймі

labels_df['object_count'] = labels_df['objects'].apply( lambda objects: len(objects) )
labels_df['object_count'].value_counts()

object_count
1    19590
0     4861
2     4012
3      172
4       59
Name: count, dtype: int64

In [14]:
labels_df['depth'].value_counts()

depth
3    28694
Name: count, dtype: int64

Всі кадри кольорові

In [15]:
labels_df[['width', 'height']].value_counts()

width  height
1280   720       28694
Name: count, dtype: int64

Всі кадри одного розміру

## Власний поділ даних на тренувальну та валідаційну вибірки

Для навчання та валідації моделі створимо власний поділ даних замість використання початкового поділу датасету. 80% відео випадковим чином буде віднесено до тренувальної вибірки, а решту 20% - до валідаційної. Поділ виконується на рівні відео, а не окремих кадрів. Це дозволяє уникнути data leakage, за якого кадри одного відео могли б одночасно потрапити до тренувальної та валідаційної вибірок.

In [16]:
video_names = list( labels_df['video_name'].unique() )
rng.shuffle(video_names)

train_size = 0.8
split_index = int( np.round( train_size * len(video_names) ) )

train_videos = video_names[:split_index]
val_videos = video_names[split_index:]

Щоб скоротити час навчання та зменшити розмір датасету, використаємо не всі доступні кадри, а випадкову вибірку максимум із 30 кадрів для кожного відео.

In [17]:
# максимальна кількість фреймів з одного відео 
max_frames = 30

new_labels_data = []

for video_name, group in labels_df.groupby("video_name"):

    if len(group) <= max_frames:
        sampled_frames = group
    else:
        sampled_frames = group.sample( n=max_frames, random_state=RANDOM_SEED )

        sampled_indices = rng.choice(group.index, size=max_frames, replace=False)
        sampled_frames = group.loc[sampled_indices]

    new_labels_data.append(sampled_frames)


new_labels_df = pd.concat(new_labels_data, ignore_index=True)

In [18]:
new_labels_df.shape

(13783, 7)

In [19]:
new_labels_df['dataset'] = new_labels_df['video_name'].apply( lambda video_name: 'train' if video_name in train_videos else 'val' )

In [20]:
new_labels_df.head()

,video_name,frame_name,width,height,depth,objects,object_count,dataset
0,bird_1,bird_1_000000.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 3, 'xmin': 405,...",1,train
1,bird_1,bird_1_000001.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 2, 'xmin': 376,...",1,train
2,bird_1,bird_1_000002.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 2, 'xmin': 334,...",1,train
3,bird_1,bird_1_000003.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 295,...",1,train
4,bird_1,bird_1_000004.jpg,1280,720,3,"[{'name': 'bird', 'difficult': 1, 'xmin': 256,...",1,train


In [21]:
new_labels_df['dataset'].value_counts()

dataset
train    11011
val       2772
Name: count, dtype: int64

In [22]:
new_labels_df[ new_labels_df['dataset'] == 'train' ]['object_count'].value_counts()

object_count
1    7313
0    2366
2    1221
3      72
4      39
Name: count, dtype: int64

In [23]:
new_labels_df[ new_labels_df['dataset'] == 'val' ]['object_count'].value_counts()

object_count
1    1801
0     589
2     367
3      15
Name: count, dtype: int64

## Збереження власної розмітки у форматі YOLO

Для кожного кадру створимо відповідний текстовий файл із розміткою у форматі YOLO. Оскільки задача передбачає виявлення лише одного класу - bird, усім об'єктам присвоюється ідентифікатор класу 0. Для кадрів, на яких відсутні анотовані об'єкти, створюються порожні файли розмітки.

In [24]:
os.makedirs('../data/processed/labels', exist_ok=True)

os.makedirs('../data/processed/labels/train', exist_ok=True)
os.makedirs('../data/processed/labels/val', exist_ok=True)


def save( row ):

    save_file_path = f"../data/processed/labels/{row.at['dataset']}/{row.at['frame_name'].replace('.jpg', '.txt')}"

    width = row.at["width"]
    height = row.at["height"]

    labels = []

    for obj in row.at['objects']:

        xmin = obj["xmin"]
        ymin = obj["ymin"]
        xmax = obj["xmax"]
        ymax = obj["ymax"]

        x_center = ((xmin + xmax) / 2) / width
        y_center = ((ymin + ymax) / 2) / height

        box_width = (xmax - xmin) / width
        box_height = (ymax - ymin) / height

        labels.append(f'0 {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}')    

    with open(save_file_path, 'w') as f:
        f.write("\n".join(labels))


new_labels_df.apply( lambda row: save(row), axis=1 )

0        None
1        None
2        None
3        None
4        None
         ... 
13778    None
13779    None
13780    None
13781    None
13782    None
Length: 13783, dtype: object

## Створення кадрів з відео

In [25]:
# ця функція була взята з документації датасету, але була модифікована 
# з врахуванням власного розбиття на тренувальну та валідаційну вибірки


def split_video(in_video_path, out_image_path):

    selected_frames = new_labels_df['frame_name'].values
    frame_dataset_dict = dict( zip(new_labels_df['frame_name'], new_labels_df['dataset']) )

    videos = os.listdir(in_video_path)

    for video_name in videos:

        video_file_path = f'{in_video_path}/{video_name}'
        video_file = cv2.VideoCapture(video_file_path)

        if not video_file.isOpened():
            print(f"Не вдалося відкрити відео: {video_file_path}")
            continue

        frame_num = 0

        while (True):

            ret, frame = video_file.read()

            if ret != True:
                break
            else:

                frame_name = f'{video_name.split(".")[0]}_{str(frame_num).zfill(6)}.jpg'

                # перевірка чи даний кадр потрапив в нову розмітку або ні
                if frame_name in selected_frames:

                    # з якої вибірки даний кадр (тестувальної чи валідаційної) в новому власному розділенні даних
                    dataset = frame_dataset_dict[frame_name]

                    frame_name_save_path = f'{out_image_path}/{dataset}/{frame_name}'
                    cv2.imwrite(frame_name_save_path, frame)
                
            frame_num +=1

In [26]:
image_path = '../data/processed/images'
os.makedirs(image_path, exist_ok=True)

os.makedirs(f'{image_path}/train', exist_ok=True)
os.makedirs(f'{image_path}/val', exist_ok=True)

split_video(in_video_path='../data/raw/FBD-SV-2024/videos/train/', out_image_path=image_path)
split_video(in_video_path='../data/raw/FBD-SV-2024/videos/val/', out_image_path=image_path)


## Створення конфігураційного файлу датасету YOLO (data.yaml)

In [27]:
data_yaml = """path: ./data/processed

train: images/train
val: images/val

names:
  0: bird"""

with open('../data/processed/data.yaml', 'w') as f:
        f.write(data_yaml)